# Notebook 09 -- Feature Extraction: Images

<div style="border-left: 4px solid #4680a7; padding: 10px 15px; margin: 10px 0; background: #4e6681;">

**Objective:** Extract visual feature representations from pet images using a pretrained `efficientnet_v2_s` backbone. Aggregate per-image embeddings into per-PetID feature vectors, augmented with image quality proxies and Vision API metadata features.

**Answers:** *"What visual features -- deep embeddings, quality signals, and metadata labels -- can be captured for adoption speed prediction?"*

</div>

| Item | Detail |
|------|--------|
| **Dependencies** | `data/cleaned/train.parquet`, `data/cleaned/test.parquet`, `data/raw/train_images/`, `data/raw/test_images/`, `data/raw/train_metadata/`, `data/raw/test_metadata/` |
| **Artifacts** | `data/features/images/v1/train.parquet`, `data/features/images/v1/test.parquet`, `data/features/images/v1/schema.json`, `artifacts/models/image_pca_v1.joblib`, `reports/feature_extraction_images.json` |
| **Model** | `efficientnet_v2_s` (torchvision, 1280-dim penultimate layer, 384x384 input) |
| **Scope** | Image embeddings, quality features, and Vision API metadata only -- no tabular, text, or cross-modality fusion |
| **Runtime** | ~3 min (GPU) / ~65 min (CPU) for embedding extraction |

---
## 1. Imports & Configuration

In [1]:
# -- Standard library & third-party imports ------------------------------------
from __future__ import annotations

import gc
import json
import random
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

warnings.filterwarnings("ignore", category=FutureWarning)

# -- Project imports -----------------------------------------------------------
from adoption_accelerator import config as cfg
from adoption_accelerator.data.ingestion import load_parquet, save_parquet
from adoption_accelerator.features.image import (
    FEATURE_DESCRIPTIONS,
    IMAGE_AUX_COLUMNS,
    IMAGE_METADATA_COLUMNS,
    IMAGE_QUALITY_COLUMNS,
    aggregate_embeddings_per_pet,
    compute_image_quality_features,
    extract_image_embeddings_batch,
    get_image_paths_for_split,
    load_pretrained_backbone,
    reduce_image_dimensions,
    _extract_split_embeddings,
    _resolve_device,
)
from adoption_accelerator.features.metadata import aggregate_metadata_features
from adoption_accelerator.features.registry import (
    build_column_descriptors,
    compute_config_hash,
    save_feature_schema,
)
from adoption_accelerator.utils.logging import setup_logging

setup_logging()

# -- Reproducibility -----------------------------------------------------------
SEED = cfg.SEED
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:.4f}".format)

# -- Image feature configuration -----------------------------------------------
IMAGE_CONFIG = {
    "model_name": "efficientnet_v2_s",
    "model_source": "torchvision",
    "input_size": 384,
    "batch_size": 32,
    "device": None,             # auto-detect (cuda > mps > cpu)
    "apply_pca": True,
    "pca_components": 100,
    "feature_dim": 1280,
    "aggregation": "mean",      # mean pooling across images per PetID
}

# -- Output paths --------------------------------------------------------------
FEATURES_DIR = cfg.DATA_FEATURES / "images" / "v1"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

REPORT_PATH = cfg.REPORTS_DIR / "feature_extraction_images.json"

# -- Resolve device ------------------------------------------------------------
DEVICE = _resolve_device(IMAGE_CONFIG["device"])

print("Imports and configuration loaded.")
print(f"   Seed          : {SEED}")
print(f"   Model         : {IMAGE_CONFIG['model_name']}")
print(f"   Input size    : {IMAGE_CONFIG['input_size']}x{IMAGE_CONFIG['input_size']}")
print(f"   Feature dim   : {IMAGE_CONFIG['feature_dim']}")
print(f"   Batch size    : {IMAGE_CONFIG['batch_size']}")
print(f"   Aggregation   : {IMAGE_CONFIG['aggregation']}")
print(f"   PCA enabled   : {IMAGE_CONFIG['apply_pca']} ({IMAGE_CONFIG['pca_components']} components)")
print(f"   Device        : {DEVICE}")
print(f"   Features dir  : {FEATURES_DIR}")
print(f"   Report path   : {REPORT_PATH}")

Imports and configuration loaded.
   Seed          : 42
   Model         : efficientnet_v2_s
   Input size    : 384x384
   Feature dim   : 1280
   Batch size    : 32
   Aggregation   : mean
   PCA enabled   : True (100 components)
   Device        : cpu
   Features dir  : C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\images\v1
   Report path   : C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\reports\feature_extraction_images.json


---
## 2. Load Cleaned Data

In [2]:
# -- Load cleaned train and test DataFrames ------------------------------------
train = load_parquet(cfg.DATA_CLEANED / "train.parquet")
test = load_parquet(cfg.DATA_CLEANED / "test.parquet")

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
print(f"\nTrain image info:")
print(f"   PetID    : {train['PetID'].nunique():,} unique")
print(f"   PhotoAmt : mean={train['PhotoAmt'].mean():.1f}, max={train['PhotoAmt'].max()}, sum={train['PhotoAmt'].sum():,}")
print(f"\nTest image info:")
print(f"   PetID    : {test['PetID'].nunique():,} unique")
print(f"   PhotoAmt : mean={test['PhotoAmt'].mean():.1f}, max={test['PhotoAmt'].max()}, sum={test['PhotoAmt'].sum():,}")

12:45:05  INFO      Loaded Parquet: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\cleaned\train.parquet (14993 rows, 25 cols)
12:45:05  INFO      Loaded Parquet: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\cleaned\test.parquet (3972 rows, 24 cols)


Train shape : (14993, 25)
Test shape  : (3972, 24)

Train image info:
   PetID    : 14,993 unique
   PhotoAmt : mean=3.9, max=30.0, sum=58,311.0

Test image info:
   PetID    : 3,972 unique
   PhotoAmt : mean=3.6, max=30.0, sum=14,465.0


In [3]:
# -- Validation Gate G09-1: Required columns present ---------------------------
EXPECTED_TRAIN_ROWS = 14_993
EXPECTED_TEST_ROWS = 3_972

assert "PetID" in train.columns and "PetID" in test.columns, "G09-1 FAIL: PetID missing"
assert "PhotoAmt" in train.columns and "PhotoAmt" in test.columns, "G09-1 FAIL: PhotoAmt missing"
assert train.shape[0] == EXPECTED_TRAIN_ROWS, (
    f"G09-1 FAIL: Expected {EXPECTED_TRAIN_ROWS} train rows, got {train.shape[0]}"
)
assert test.shape[0] == EXPECTED_TEST_ROWS, (
    f"G09-1 FAIL: Expected {EXPECTED_TEST_ROWS} test rows, got {test.shape[0]}"
)

print("G09-1 PASS: Cleaned data loaded with PetID and PhotoAmt columns.")
print(f"   Train: {train.shape[0]:,} rows x {train.shape[1]} cols")
print(f"   Test : {test.shape[0]:,} rows x {test.shape[1]} cols")

G09-1 PASS: Cleaned data loaded with PetID and PhotoAmt columns.
   Train: 14,993 rows x 25 cols
   Test : 3,972 rows x 24 cols


---
## 3. Image Path Inventory

<div style="border-left: 4px solid #f39c12; padding: 8px 12px; margin: 8px 0; background: #5e5535;">

**Strategy:** For each PetID, enumerate available images on disk. Cross-reference with `PhotoAmt` to detect consistency issues. Log image coverage statistics.

</div>

In [4]:
# -- Resolve image paths for both splits ---------------------------------------
train_pet_ids = train["PetID"].tolist()
test_pet_ids = test["PetID"].tolist()

train_image_paths = get_image_paths_for_split(train_pet_ids, "train")
test_image_paths = get_image_paths_for_split(test_pet_ids, "test")

# Image inventory statistics
train_img_counts = [len(v) for v in train_image_paths.values()]
test_img_counts = [len(v) for v in test_image_paths.values()]

train_total_images = sum(train_img_counts)
test_total_images = sum(test_img_counts)
train_with_images = sum(1 for c in train_img_counts if c > 0)
test_with_images = sum(1 for c in test_img_counts if c > 0)

print("Image path inventory:")
print(f"   Train: {train_total_images:,} images across {train_with_images:,} / {len(train_pet_ids):,} PetIDs")
print(f"   Test : {test_total_images:,} images across {test_with_images:,} / {len(test_pet_ids):,} PetIDs")
print(f"\nTrain image count distribution:")
print(f"   Mean  : {np.mean(train_img_counts):.1f}")
print(f"   Median: {np.median(train_img_counts):.0f}")
print(f"   Max   : {max(train_img_counts)}")
print(f"   Zero  : {sum(1 for c in train_img_counts if c == 0):,} PetIDs with no images")

12:59:48  INFO      Image path resolution (split=train): 14993 PetIDs, 14652 with images, 58311 total files
13:00:48  INFO      Image path resolution (split=test): 3972 PetIDs, 3858 with images, 14465 total files


Image path inventory:
   Train: 58,311 images across 14,652 / 14,993 PetIDs
   Test : 14,465 images across 3,858 / 3,972 PetIDs

Train image count distribution:
   Mean  : 3.9
   Median: 3
   Max   : 30
   Zero  : 341 PetIDs with no images


In [5]:
# -- Cross-reference with PhotoAmt ---------------------------------------------
photo_amt_train = train.set_index("PetID")["PhotoAmt"]
disk_counts_train = pd.Series(
    {pid: len(paths) for pid, paths in train_image_paths.items()},
    name="disk_count",
)
mismatch_train = (photo_amt_train.reindex(disk_counts_train.index) != disk_counts_train).sum()

photo_amt_test = test.set_index("PetID")["PhotoAmt"]
disk_counts_test = pd.Series(
    {pid: len(paths) for pid, paths in test_image_paths.items()},
    name="disk_count",
)
mismatch_test = (photo_amt_test.reindex(disk_counts_test.index) != disk_counts_test).sum()

print(f"PhotoAmt vs disk count mismatches:")
print(f"   Train: {mismatch_train:,} PetIDs")
print(f"   Test : {mismatch_test:,} PetIDs")

PhotoAmt vs disk count mismatches:
   Train: 0 PetIDs
   Test : 0 PetIDs


---
## 4. Load Pretrained Backbone

<div style="border-left: 4px solid #f39c12; padding: 8px 12px; margin: 8px 0; background: #5e5535;">

**Model:** `efficientnet_v2_s` -- EfficientNet V2 Small via torchvision. Compound-scaled CNN producing 1,280-dimensional feature vectors from the penultimate layer. Classification head removed; model set to eval mode.

</div>

In [6]:
# -- Load pretrained backbone --------------------------------------------------
model, transform, actual_feature_dim = load_pretrained_backbone(
    model_name=IMAGE_CONFIG["model_name"],
    device=DEVICE,
)

# Validation Gate G09-2: Model loaded correctly
assert model is not None, "G09-2 FAIL: Model is None"
assert not model.training, "G09-2 FAIL: Model not in eval mode"
assert actual_feature_dim == IMAGE_CONFIG["feature_dim"], (
    f"G09-2 FAIL: Feature dim mismatch: {actual_feature_dim} != {IMAGE_CONFIG['feature_dim']}"
)

print(f"G09-2 PASS: Pretrained model loaded successfully.")
print(f"   Model       : {IMAGE_CONFIG['model_name']}")
print(f"   Feature dim : {actual_feature_dim}")
print(f"   Eval mode   : {not model.training}")
print(f"   Device      : {DEVICE}")

# Quick sanity check with a dummy tensor
dummy = torch.randn(1, 3, IMAGE_CONFIG["input_size"], IMAGE_CONFIG["input_size"]).to(DEVICE)
with torch.no_grad():
    dummy_out = model(dummy)
print(f"   Sanity check: input={list(dummy.shape)} -> output={list(dummy_out.shape)}")
del dummy, dummy_out

13:01:27  INFO      Loading pretrained model: efficientnet_v2_s (device=cpu)


Downloading: "https://download.pytorch.org/models/efficientnet_v2_s-dd5fe13b.pth" to C:\Users\cliente/.cache\torch\hub\checkpoints\efficientnet_v2_s-dd5fe13b.pth


100%|██████████| 82.7M/82.7M [00:00<00:00, 93.6MB/s]
13:01:29  INFO      Model loaded: efficientnet_v2_s | input=384x384 | feature_dim=1280 | device=cpu


G09-2 PASS: Pretrained model loaded successfully.
   Model       : efficientnet_v2_s
   Feature dim : 1280
   Eval mode   : True
   Device      : cpu
   Sanity check: input=[1, 3, 384, 384] -> output=[1, 1280]


---
## 5. Extract Image Embeddings -- Train

<div style="border-left: 4px solid #8e44ad; padding: 8px 12px; margin: 8px 0; background: #4a3554;">

Process all train images in batches through the pretrained backbone. Extract penultimate-layer activations. Handle I/O failures gracefully with zero-vector fallback. Track progress and timing.

</div>

In [7]:
# -- Extract train image embeddings --------------------------------------------
t0 = time.perf_counter()
train_emb_dict = _extract_split_embeddings(
    pet_ids=train_pet_ids,
    pet_image_paths=train_image_paths,
    model=model,
    transform=transform,
    batch_size=IMAGE_CONFIG["batch_size"],
    device=DEVICE,
    feature_dim=actual_feature_dim,
)
train_extract_time = time.perf_counter() - t0

train_emb_pet_count = len(train_emb_dict)
train_emb_img_count = sum(len(v) for v in train_emb_dict.values())

print(f"Train embedding extraction complete:")
print(f"   Time         : {train_extract_time:.1f}s")
print(f"   PetIDs       : {train_emb_pet_count:,} (with at least 1 image)")
print(f"   Images       : {train_emb_img_count:,} successfully embedded")
print(f"   Throughput   : {train_emb_img_count / max(train_extract_time, 0.1):.0f} images/sec")
if train_total_images > 0:
    fail_rate = 1.0 - (train_emb_img_count / train_total_images)
    print(f"   Failure rate : {fail_rate:.2%}")

13:02:14  INFO      Extracting embeddings for 58311 images ...


Train embedding extraction complete:
   Time         : 3249.9s
   PetIDs       : 14,652 (with at least 1 image)
   Images       : 58,311 successfully embedded
   Throughput   : 18 images/sec
   Failure rate : 0.00%


---
## 6. Aggregate Train Embeddings

In [8]:
# -- Aggregate per-PetID (train) -----------------------------------------------
train_agg, train_has_img, train_photo_ct = aggregate_embeddings_per_pet(
    embeddings_dict=train_emb_dict,
    pet_ids=train_pet_ids,
    embedding_dim=actual_feature_dim,
    strategy=IMAGE_CONFIG["aggregation"],
)

print(f"Train aggregation ({IMAGE_CONFIG['aggregation']} pooling):")
print(f"   Shape         : {train_agg.shape}")
print(f"   With images   : {int(train_has_img.sum()):,}")
print(f"   Without images: {int((train_has_img == 0).sum()):,}")
print(f"   Memory        : {train_agg.nbytes / 1_048_576:.1f} MB")

# Free per-image embeddings
del train_emb_dict
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

13:57:25  INFO      Aggregated embeddings (mean): 14993 PetIDs, 14652 with images, 341 without


Train aggregation (mean pooling):
   Shape         : (14993, 1280)
   With images   : 14,652
   Without images: 341
   Memory        : 73.2 MB


---
## 7. Extract Image Embeddings -- Test

In [9]:
# -- Extract test image embeddings ---------------------------------------------
t0 = time.perf_counter()
test_emb_dict = _extract_split_embeddings(
    pet_ids=test_pet_ids,
    pet_image_paths=test_image_paths,
    model=model,
    transform=transform,
    batch_size=IMAGE_CONFIG["batch_size"],
    device=DEVICE,
    feature_dim=actual_feature_dim,
)
test_extract_time = time.perf_counter() - t0

test_emb_pet_count = len(test_emb_dict)
test_emb_img_count = sum(len(v) for v in test_emb_dict.values())

print(f"Test embedding extraction complete:")
print(f"   Time         : {test_extract_time:.1f}s")
print(f"   PetIDs       : {test_emb_pet_count:,} (with at least 1 image)")
print(f"   Images       : {test_emb_img_count:,} successfully embedded")
print(f"   Throughput   : {test_emb_img_count / max(test_extract_time, 0.1):.0f} images/sec")

13:57:33  INFO      Extracting embeddings for 14465 images ...


Test embedding extraction complete:
   Time         : 735.3s
   PetIDs       : 3,858 (with at least 1 image)
   Images       : 14,465 successfully embedded
   Throughput   : 20 images/sec


---
## 8. Aggregate Test Embeddings

In [10]:
# -- Aggregate per-PetID (test) ------------------------------------------------
test_agg, test_has_img, test_photo_ct = aggregate_embeddings_per_pet(
    embeddings_dict=test_emb_dict,
    pet_ids=test_pet_ids,
    embedding_dim=actual_feature_dim,
    strategy=IMAGE_CONFIG["aggregation"],
)

print(f"Test aggregation ({IMAGE_CONFIG['aggregation']} pooling):")
print(f"   Shape         : {test_agg.shape}")
print(f"   With images   : {int(test_has_img.sum()):,}")
print(f"   Without images: {int((test_has_img == 0).sum()):,}")

# Free per-image embeddings and model
del test_emb_dict, model
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

14:09:48  INFO      Aggregated embeddings (mean): 3972 PetIDs, 3858 with images, 114 without


Test aggregation (mean pooling):
   Shape         : (3972, 1280)
   With images   : 3,858
   Without images: 114


---
## 9. Dimensionality Reduction (PCA)

<div style="border-left: 4px solid #f39c12; padding: 8px 12px; margin: 8px 0; background: #5e5535;">

**Strategy:** Fit PCA on train aggregated embeddings and apply to both splits. Reduces 1,280 dimensions to 100 components while preserving the majority of variance. The fitted PCA object is saved for inference reproducibility.

</div>

In [11]:
# -- PCA dimensionality reduction ----------------------------------------------
effective_dim = actual_feature_dim
pca_obj = None
pca_explained_var = None

if IMAGE_CONFIG["apply_pca"]:
    pca_components = IMAGE_CONFIG["pca_components"]
    train_agg, test_agg, pca_obj, pca_explained_var = reduce_image_dimensions(
        train_agg, test_agg, n_components=pca_components,
    )
    effective_dim = pca_components

    print(f"PCA dimensionality reduction:")
    print(f"   Components     : {pca_components}")
    print(f"   Explained var  : {pca_explained_var:.4f} ({pca_explained_var * 100:.2f}%)")
    print(f"   Train shape    : {train_agg.shape}")
    print(f"   Test shape     : {test_agg.shape}")
    print(f"   Compression    : {actual_feature_dim} -> {pca_components} ({pca_components / actual_feature_dim:.1%} of original)")
else:
    print(f"PCA disabled. Embedding dim = {effective_dim}")

14:09:54  INFO      Fitting PCA with n_components=100 on train image embeddings ...
14:09:54  INFO      PCA complete: 1280 -> 100 dimensions, explained variance=0.8274


PCA dimensionality reduction:
   Components     : 100
   Explained var  : 0.8274 (82.74%)
   Train shape    : (14993, 100)
   Test shape     : (3972, 100)
   Compression    : 1280 -> 100 (7.8% of original)


In [12]:
# -- Save PCA model artifact (if fitted) ---------------------------------------
if pca_obj is not None:
    import joblib
    pca_path = cfg.ARTIFACTS_MODELS / "image_pca_v1.joblib"
    pca_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(pca_obj, pca_path)
    print(f"PCA model saved: {pca_path}")

PCA model saved: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\artifacts\models\image_pca_v1.joblib


---
## 10. Vision API Metadata Features

In [13]:
# -- Aggregate Vision API metadata features ------------------------------------
print("Loading and aggregating train metadata JSONs...")
meta_train = aggregate_metadata_features(split="train")

print(f"\nLoading and aggregating test metadata JSONs...")
meta_test = aggregate_metadata_features(split="test")

print(f"\nMetadata feature coverage:")
meta_coverage_train = meta_train["PetID"].isin(train["PetID"]).sum() / len(train)
meta_coverage_test = meta_test["PetID"].isin(test["PetID"]).sum() / len(test)
print(f"   Train: {meta_coverage_train:.2%} ({len(meta_train):,} PetIDs with metadata / {len(train):,} total)")
print(f"   Test : {meta_coverage_test:.2%} ({len(meta_test):,} PetIDs with metadata / {len(test):,} total)")

Loading and aggregating train metadata JSONs...


14:09:56  INFO      Found 58311 metadata JSON files in C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\raw\train_metadata
14:11:09  INFO        ... loaded 10000 / 58311 metadata JSONs
14:12:31  INFO        ... loaded 20000 / 58311 metadata JSONs
14:13:47  INFO        ... loaded 30000 / 58311 metadata JSONs
14:15:01  INFO        ... loaded 40000 / 58311 metadata JSONs
14:16:26  INFO        ... loaded 50000 / 58311 metadata JSONs
14:17:27  INFO      Aggregated metadata features for split='train': 14652 PetIDs x 7 cols



Loading and aggregating test metadata JSONs...


14:17:27  INFO      Found 14465 metadata JSON files in C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\raw\test_metadata
14:18:41  INFO        ... loaded 10000 / 14465 metadata JSONs
14:19:11  INFO      Aggregated metadata features for split='test': 3858 PetIDs x 7 cols



Metadata feature coverage:
   Train: 97.73% (14,652 PetIDs with metadata / 14,993 total)
   Test : 97.13% (3,858 PetIDs with metadata / 3,972 total)


In [14]:
# -- Preview metadata features -------------------------------------------------
meta_feature_cols = [c for c in meta_train.columns if c != "PetID"]
print(f"Metadata feature columns ({len(meta_feature_cols)}):")
for col in meta_feature_cols:
    print(f"   {col}")

display(
    meta_train[meta_feature_cols].describe()
    .T.style.format("{:.4f}")
    .set_caption("Train Vision API Metadata Feature Statistics")
)

Metadata feature columns (6):
   meta_label_count_mean
   meta_top_label_score_mean
   meta_dominant_color_count_mean
   meta_avg_brightness_mean
   meta_color_diversity_mean
   meta_crop_confidence_mean


,count,mean,std,min,25%,50%,75%,max
meta_label_count_mean,14652.0000,9.3828,0.9156,0.5000,9.0000,9.8000,10.0000,10.0000
meta_top_label_score_mean,14652.0000,0.9695,0.0256,0.2600,0.9559,0.9738,0.9904,0.9953
meta_dominant_color_count_mean,14652.0000,9.9788,0.1846,5.6667,10.0000,10.0000,10.0000,10.0000
meta_avg_brightness_mean,14652.0000,113.0891,18.4559,43.6000,100.9395,113.0705,124.7691,194.1700
meta_color_diversity_mean,14652.0000,50.7741,9.5084,15.7100,44.5805,51.0600,57.3081,85.7500
meta_crop_confidence_mean,14652.0000,0.8022,0.0220,0.3000,0.8000,0.8000,0.8000,1.0000


---
## 11. Image Quality Features

In [15]:
# -- Compute image quality features --------------------------------------------
print("Computing image quality features for train...")
t0 = time.perf_counter()
train_quality = compute_image_quality_features(train_image_paths, train_pet_ids)
train_quality_time = time.perf_counter() - t0
print(f"   Train quality features: {train_quality.shape} ({train_quality_time:.1f}s)")

print("Computing image quality features for test...")
t0 = time.perf_counter()
test_quality = compute_image_quality_features(test_image_paths, test_pet_ids)
test_quality_time = time.perf_counter() - t0
print(f"   Test quality features: {test_quality.shape} ({test_quality_time:.1f}s)")

print(f"\nQuality feature summary (train):")
for col in IMAGE_QUALITY_COLUMNS:
    vals = train_quality[col]
    print(f"   {col:<25s}: mean={vals.mean():.2f}, std={vals.std():.2f}, min={vals.min():.2f}, max={vals.max():.2f}")

Computing image quality features for train...


14:21:16  INFO      Computed image quality features for 14993 PetIDs


   Train quality features: (14993, 3) (124.4s)
Computing image quality features for test...


14:21:46  INFO      Computed image quality features for 3972 PetIDs


   Test quality features: (3972, 3) (30.1s)

Quality feature summary (train):
   mean_image_brightness    : mean=117.11, std=29.03, min=0.00, max=229.23
   mean_blur_score          : mean=1067.53, std=1344.22, min=0.00, max=26934.59
   image_size_std           : mean=15159.54, std=32510.76, min=0.00, max=2210342.53


---
## 12. Assemble Image Feature Matrices

<div style="border-left: 4px solid #27ae60; padding: 8px 12px; margin: 8px 0; background: #2d4a37;">

Merge aggregated embeddings, auxiliary features (has_image, photo_count), image quality proxies, and Vision API metadata into a single DataFrame per split, keyed on `PetID`.

</div>

In [16]:
# -- Build embedding DataFrames ------------------------------------------------
emb_col_names = [f"img_emb_{i}" for i in range(effective_dim)]

train_emb_df = pd.DataFrame(train_agg.astype(np.float32), columns=emb_col_names)
test_emb_df = pd.DataFrame(test_agg.astype(np.float32), columns=emb_col_names)

print(f"Embedding DataFrames:")
print(f"   Train: {train_emb_df.shape}")
print(f"   Test : {test_emb_df.shape}")

Embedding DataFrames:
   Train: (14993, 100)
   Test : (3972, 100)


In [17]:
# -- Assemble TRAIN feature matrix ---------------------------------------------
train_features = pd.DataFrame({
    "PetID": train_pet_ids,
    "has_image_embedding": train_has_img,
    "actual_photo_count": train_photo_ct,
})

# Add embeddings
train_features = pd.concat([train_features, train_emb_df.reset_index(drop=True)], axis=1)

# Add quality features
for col in IMAGE_QUALITY_COLUMNS:
    train_features[col] = train_quality[col].values

# Add metadata features
train_features = train_features.merge(
    meta_train, on="PetID", how="left",
)
for col in IMAGE_METADATA_COLUMNS:
    if col in train_features.columns:
        train_features[col] = train_features[col].fillna(0.0).astype(np.float32)

print(f"Train feature matrix: {train_features.shape}")
print(f"   Null values: {int(train_features.isna().sum().sum())}")

Train feature matrix: (14993, 112)
   Null values: 0


In [18]:
# -- Assemble TEST feature matrix ----------------------------------------------
test_features = pd.DataFrame({
    "PetID": test_pet_ids,
    "has_image_embedding": test_has_img,
    "actual_photo_count": test_photo_ct,
})

# Add embeddings
test_features = pd.concat([test_features, test_emb_df.reset_index(drop=True)], axis=1)

# Add quality features
for col in IMAGE_QUALITY_COLUMNS:
    test_features[col] = test_quality[col].values

# Add metadata features
test_features = test_features.merge(
    meta_test, on="PetID", how="left",
)
for col in IMAGE_METADATA_COLUMNS:
    if col in test_features.columns:
        test_features[col] = test_features[col].fillna(0.0).astype(np.float32)

print(f"Test feature matrix: {test_features.shape}")
print(f"   Null values: {int(test_features.isna().sum().sum())}")

Test feature matrix: (3972, 112)
   Null values: 0


---
## 13. Feature Matrix Inspection

In [19]:
# -- Feature matrix overview ---------------------------------------------------
feature_cols_no_id = [c for c in train_features.columns if c != "PetID"]
n_aux = len(IMAGE_AUX_COLUMNS)
n_emb = len(emb_col_names)
n_quality = len(IMAGE_QUALITY_COLUMNS)
n_meta = sum(1 for c in IMAGE_METADATA_COLUMNS if c in train_features.columns)

print(f"Image feature matrix structure:")
print(f"   PetID              : 1 column (index key)")
print(f"   Auxiliary features  : {n_aux} columns (has_image_embedding, actual_photo_count)")
print(f"   Embedding dims      : {n_emb} columns (img_emb_0 .. img_emb_{n_emb - 1})")
print(f"   Quality features    : {n_quality} columns")
print(f"   Metadata features   : {n_meta} columns")
print(f"   Total features      : {len(feature_cols_no_id)}")
print(f"\n   Train shape: {train_features.shape}")
print(f"   Test shape : {test_features.shape}")

Image feature matrix structure:
   PetID              : 1 column (index key)
   Auxiliary features  : 2 columns (has_image_embedding, actual_photo_count)
   Embedding dims      : 100 columns (img_emb_0 .. img_emb_99)
   Quality features    : 3 columns
   Metadata features   : 6 columns
   Total features      : 111

   Train shape: (14993, 112)
   Test shape : (3972, 112)


In [20]:
# -- Data types summary --------------------------------------------------------
print("Data type counts:")
print(train_features.dtypes.value_counts().to_string())
print(f"\nMemory usage:")
print(f"   Train: {train_features.memory_usage(deep=True).sum() / 1_048_576:.2f} MB")
print(f"   Test : {test_features.memory_usage(deep=True).sum() / 1_048_576:.2f} MB")

Data type counts:
float32    106
float64      3
int32        2
str          1

Memory usage:
   Train: 6.76 MB
   Test : 1.79 MB


In [21]:
# -- Descriptive statistics for non-embedding features -------------------------
non_emb_cols = IMAGE_AUX_COLUMNS + IMAGE_QUALITY_COLUMNS + [c for c in IMAGE_METADATA_COLUMNS if c in train_features.columns]
display(
    train_features[non_emb_cols].describe()
    .T.style.format("{:.4f}")
    .set_caption("Train Image Non-Embedding Features -- Descriptive Statistics")
)

,count,mean,std,min,25%,50%,75%,max
has_image_embedding,14993.0000,0.9773,0.1491,0.0000,1.0000,1.0000,1.0000,1.0000
actual_photo_count,14993.0000,3.8892,3.4878,0.0000,2.0000,3.0000,5.0000,30.0000
mean_image_brightness,14993.0000,117.1073,29.0333,0.0000,104.7000,119.3000,133.2700,229.2300
mean_blur_score,14993.0000,1067.5308,1344.2206,0.0000,303.8700,631.2500,1288.8500,26934.5900
image_size_std,14993.0000,15159.5366,32510.7599,0.0000,0.0000,0.0000,18149.3500,2210342.5300
meta_label_count_mean,14993.0000,9.1694,1.6661,0.0000,9.0000,9.7500,10.0000,10.0000
meta_top_label_score_mean,14993.0000,0.9474,0.1467,0.0000,0.9544,0.9730,0.9902,0.9953
meta_dominant_color_count_mean,14993.0000,9.7518,1.4989,0.0000,10.0000,10.0000,10.0000,10.0000
meta_avg_brightness_mean,14993.0000,110.5170,24.8426,0.0000,100.0956,112.4900,124.4700,194.1700
meta_color_diversity_mean,14993.0000,49.6193,12.0689,0.0000,44.0050,50.8067,57.1700,85.7500


In [22]:
# -- Embedding statistics (summary over dimensions) ----------------------------
train_emb_array = train_features[emb_col_names].values
print("Embedding dimension statistics (across all train samples):")
print(f"   Mean of means : {train_emb_array.mean():.6f}")
print(f"   Mean of stds  : {train_emb_array.std(axis=0).mean():.6f}")
print(f"   Min value     : {train_emb_array.min():.6f}")
print(f"   Max value     : {train_emb_array.max():.6f}")
print(f"   Norm range    : [{np.linalg.norm(train_emb_array, axis=1).min():.4f}, {np.linalg.norm(train_emb_array, axis=1).max():.4f}]")

Embedding dimension statistics (across all train samples):
   Mean of means : 0.000000
   Mean of stds  : 0.541191
   Min value     : -6.172308
   Max value     : 6.917758
   Norm range    : [3.4838, 11.8538]


In [23]:
# -- Train vs Test column alignment check --------------------------------------
train_cols = set(train_features.columns)
test_cols = set(test_features.columns)

only_train = train_cols - test_cols
only_test = test_cols - train_cols

print("Column alignment check:")
if not only_train and not only_test:
    print("   PASS: Train and test have identical column sets.")
else:
    if only_train:
        print(f"   Only in train: {only_train}")
    if only_test:
        print(f"   Only in test : {only_test}")

print(f"\nTrain: {train_features.shape} | Test: {test_features.shape}")

Column alignment check:
   PASS: Train and test have identical column sets.

Train: (14993, 112) | Test: (3972, 112)


---
## 14. Persist Feature Matrices & Schema

<div style="border-left: 4px solid #8e44ad; padding: 8px 12px; margin: 8px 0; background: #4a3554;">

Feature matrices are saved to `data/features/images/v1/` as versioned Parquet files. A `schema.json` records column metadata, model name, embedding dimensionality, aggregation strategy, PCA configuration, row counts, and the config hash for reproducibility tracking.

</div>

In [24]:
# -- Save feature matrices to Parquet ------------------------------------------
train_path = save_parquet(train_features, FEATURES_DIR / "train.parquet")
test_path = save_parquet(test_features, FEATURES_DIR / "test.parquet")

print(f"Train features saved: {train_path}")
print(f"Test features saved : {test_path}")

14:21:46  INFO      Saved Parquet: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\images\v1\train.parquet (14993 rows, 8.70 MB)
14:21:46  INFO      Saved Parquet: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\images\v1\test.parquet (3972 rows, 2.25 MB)


Train features saved: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\images\v1\train.parquet
Test features saved : C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\images\v1\test.parquet


In [25]:
# -- Generate and save schema.json ---------------------------------------------
# Build descriptions for embedding columns
all_descriptions = dict(FEATURE_DESCRIPTIONS)
for col in emb_col_names:
    dim_idx = col.split("_")[-1]
    pca_note = f" (PCA component {dim_idx})" if IMAGE_CONFIG["apply_pca"] else f" (raw dim {dim_idx})"
    all_descriptions[col] = f"Image embedding dimension{pca_note} (efficientnet_v2_s)"

column_descriptors = build_column_descriptors(
    train_features[feature_cols_no_id],
    source="image",
    descriptions=all_descriptions,
)

config_for_hash = {
    "version": "v1",
    "modality": "image",
    "seed": SEED,
    "model_name": IMAGE_CONFIG["model_name"],
    "feature_dim": IMAGE_CONFIG["feature_dim"],
    "apply_pca": IMAGE_CONFIG["apply_pca"],
    "pca_components": IMAGE_CONFIG["pca_components"],
    "aggregation": IMAGE_CONFIG["aggregation"],
    "input_size": IMAGE_CONFIG["input_size"],
    "feature_columns": feature_cols_no_id,
}

schema_metadata = {
    "version": "v1",
    "modality": "image",
    "model_name": IMAGE_CONFIG["model_name"],
    "config_hash": compute_config_hash(config_for_hash),
    "n_rows_train": len(train_features),
    "n_rows_test": len(test_features),
    "n_features": len(feature_cols_no_id),
    "seed": SEED,
    "notes": (
        f"Image features: {n_aux} auxiliary + {n_emb} embedding dims "
        f"({'PCA ' + str(IMAGE_CONFIG['pca_components']) if IMAGE_CONFIG['apply_pca'] else 'raw ' + str(actual_feature_dim)}) "
        f"+ {n_quality} quality + {n_meta} metadata features. "
        f"Backbone: efficientnet_v2_s (torchvision). "
        f"Aggregation: {IMAGE_CONFIG['aggregation']} pooling."
    ),
}

schema_path = save_feature_schema(column_descriptors, schema_metadata, FEATURES_DIR / "schema.json")
print(f"Schema saved: {schema_path}")

14:21:46  INFO      Saved feature schema: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\images\v1\schema.json (111 columns, modality=image)


Schema saved: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\data\features\images\v1\schema.json


In [26]:
# -- Save extraction report ----------------------------------------------------
report = {
    "notebook": "09_feature_extraction_images",
    "model_name": IMAGE_CONFIG["model_name"],
    "model_source": IMAGE_CONFIG["model_source"],
    "input_size": IMAGE_CONFIG["input_size"],
    "raw_feature_dim": actual_feature_dim,
    "effective_embedding_dim": effective_dim,
    "apply_pca": IMAGE_CONFIG["apply_pca"],
    "pca_components": IMAGE_CONFIG["pca_components"] if IMAGE_CONFIG["apply_pca"] else None,
    "pca_explained_variance": round(pca_explained_var, 4) if pca_explained_var else None,
    "aggregation": IMAGE_CONFIG["aggregation"],
    "device": DEVICE,
    "train_shape": list(train_features.shape),
    "test_shape": list(test_features.shape),
    "n_features": len(feature_cols_no_id),
    "n_embedding_dims": n_emb,
    "n_quality_features": n_quality,
    "n_metadata_features": n_meta,
    "n_auxiliary_features": n_aux,
    "feature_columns": feature_cols_no_id,
    "image_inventory": {
        "train_total_images": train_total_images,
        "test_total_images": test_total_images,
        "train_petids_with_images": train_with_images,
        "test_petids_with_images": test_with_images,
    },
    "extraction_time_seconds": {
        "train": round(train_extract_time, 1),
        "test": round(test_extract_time, 1),
    },
    "metadata_coverage": {
        "train": round(float(meta_coverage_train), 4),
        "test": round(float(meta_coverage_test), 4),
    },
    "config_hash": schema_metadata["config_hash"],
}

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False, default=str)

print(f"Extraction report saved: {REPORT_PATH}")

Extraction report saved: C:\Projects\Personal_Projects\AI_Adoption_Acceleration\adoption_accelerator\reports\feature_extraction_images.json


---
## 15. Validation Gate -- G09

In [27]:
#  VALIDATION GATE -- G09
# ======================================================================

gate_results = []

# G09-1: Cleaned train/test loaded with PetID and PhotoAmt columns
try:
    assert "PetID" in train.columns and "PhotoAmt" in train.columns
    assert "PetID" in test.columns and "PhotoAmt" in test.columns
    gate_results.append(("G09-1", "PASS", "Cleaned data loaded with PetID and PhotoAmt columns"))
except AssertionError as e:
    gate_results.append(("G09-1", "FAIL", str(e)))

# G09-2: Pretrained model loaded successfully
try:
    assert actual_feature_dim == IMAGE_CONFIG["feature_dim"], (
        f"Feature dim mismatch: {actual_feature_dim} != {IMAGE_CONFIG['feature_dim']}"
    )
    gate_results.append(("G09-2", "PASS", f"Pretrained model loaded: {IMAGE_CONFIG['model_name']}, dim={actual_feature_dim}"))
except AssertionError as e:
    gate_results.append(("G09-2", "FAIL", str(e)))

# G09-3: No NaN values in embedding columns
try:
    train_emb_nans = int(train_features[emb_col_names].isna().sum().sum())
    test_emb_nans = int(test_features[emb_col_names].isna().sum().sum())
    assert train_emb_nans == 0, f"Train has {train_emb_nans} NaN embedding values"
    assert test_emb_nans == 0, f"Test has {test_emb_nans} NaN embedding values"
    gate_results.append(("G09-3", "PASS", "No NaN values in embedding columns (zero-vector fallback active)"))
except AssertionError as e:
    gate_results.append(("G09-3", "FAIL", str(e)))

# G09-4: Train and test feature matrices have identical column sets
try:
    assert set(train_features.columns) == set(test_features.columns), (
        f"Column mismatch: train_only={train_cols - test_cols}, test_only={test_cols - train_cols}"
    )
    gate_results.append(("G09-4", "PASS", "Train and test have identical column sets"))
except AssertionError as e:
    gate_results.append(("G09-4", "FAIL", str(e)))

# G09-5: Row counts preserved
try:
    assert len(train_features) == EXPECTED_TRAIN_ROWS, (
        f"Train rows: {len(train_features)} != {EXPECTED_TRAIN_ROWS}"
    )
    assert len(test_features) == EXPECTED_TEST_ROWS, (
        f"Test rows: {len(test_features)} != {EXPECTED_TEST_ROWS}"
    )
    gate_results.append(("G09-5", "PASS", f"Row counts preserved: {EXPECTED_TRAIN_ROWS:,} / {EXPECTED_TEST_ROWS:,}"))
except AssertionError as e:
    gate_results.append(("G09-5", "FAIL", str(e)))

# G09-6: PetID uniqueness maintained
try:
    assert train_features["PetID"].is_unique, "Train PetID not unique"
    assert test_features["PetID"].is_unique, "Test PetID not unique"
    gate_results.append(("G09-6", "PASS", "PetID uniqueness maintained"))
except AssertionError as e:
    gate_results.append(("G09-6", "FAIL", str(e)))

# G09-7: Image extraction error rate < 1%
try:
    if train_total_images > 0:
        train_fail_rate = 1.0 - (train_emb_img_count / train_total_images)
    else:
        train_fail_rate = 0.0
    if train_fail_rate < 0.01:
        gate_results.append(("G09-7", "PASS", f"Image extraction error rate: {train_fail_rate:.4%}"))
    else:
        gate_results.append(("G09-7", "WARN", f"Image extraction error rate: {train_fail_rate:.4%} (>1%)"))
except Exception as e:
    gate_results.append(("G09-7", "WARN", str(e)))

# G09-8: PCA explained variance >= 90%
try:
    if IMAGE_CONFIG["apply_pca"] and pca_explained_var is not None:
        if pca_explained_var >= 0.90:
            gate_results.append(("G09-8", "PASS", f"PCA explained variance: {pca_explained_var:.4f} (>= 0.90)"))
        else:
            gate_results.append(("G09-8", "WARN", f"PCA explained variance: {pca_explained_var:.4f} (< 0.90)"))
    else:
        gate_results.append(("G09-8", "PASS", "PCA not applied -- gate not applicable"))
except Exception as e:
    gate_results.append(("G09-8", "WARN", str(e)))

# G09-9: No target leakage
try:
    assert "AdoptionSpeed" not in train_features.columns, "AdoptionSpeed in train features"
    assert "AdoptionSpeed" not in test_features.columns, "AdoptionSpeed in test features"
    gate_results.append(("G09-9", "PASS", "No target leakage in feature matrices"))
except AssertionError as e:
    gate_results.append(("G09-9", "FAIL", str(e)))

# G09-10: Parquet files exist and are loadable
try:
    reload_train = pd.read_parquet(FEATURES_DIR / "train.parquet")
    reload_test = pd.read_parquet(FEATURES_DIR / "test.parquet")
    gate_results.append(("G09-10", "PASS", "Parquet files exist and are loadable"))
except Exception as e:
    gate_results.append(("G09-10", "FAIL", str(e)))

# G09-11: schema.json exists, valid JSON, records model name
try:
    with open(FEATURES_DIR / "schema.json", "r", encoding="utf-8") as f:
        loaded_schema = json.load(f)
    assert "columns" in loaded_schema, "schema.json missing 'columns' key"
    assert "version" in loaded_schema, "schema.json missing 'version' key"
    assert loaded_schema.get("model_name") == IMAGE_CONFIG["model_name"], (
        f"Model name mismatch in schema: {loaded_schema.get('model_name')}"
    )
    gate_results.append(("G09-11", "PASS", "schema.json exists, valid JSON, records model name"))
except Exception as e:
    gate_results.append(("G09-11", "FAIL", str(e)))

# G09-12: has_image_embedding flag correctly marks PetIDs with PhotoAmt == 0
try:
    train_no_photo = set(train.loc[train["PhotoAmt"] == 0, "PetID"].values)
    train_no_emb = set(
        train_features.loc[train_features["has_image_embedding"] == 0, "PetID"].values
    )
    # PetIDs with PhotoAmt==0 should have has_image_embedding==0
    expected_overlap = train_no_photo.issubset(train_no_emb)
    if expected_overlap:
        gate_results.append(("G09-12", "PASS", f"has_image_embedding correctly marks {len(train_no_photo)} PetIDs with PhotoAmt==0"))
    else:
        missing = train_no_photo - train_no_emb
        gate_results.append(("G09-12", "WARN", f"{len(missing)} PetIDs with PhotoAmt==0 incorrectly flagged"))
except Exception as e:
    gate_results.append(("G09-12", "WARN", str(e)))

# G09-13: Metadata features present for >= 95% of PetIDs with PhotoAmt > 0
try:
    train_with_photo = train.loc[train["PhotoAmt"] > 0, "PetID"]
    meta_coverage_actual = meta_train["PetID"].isin(train_with_photo).sum() / len(train_with_photo)
    if meta_coverage_actual >= 0.95:
        gate_results.append(("G09-13", "PASS", f"Metadata coverage: {meta_coverage_actual:.2%} of PetIDs with photos"))
    else:
        gate_results.append(("G09-13", "WARN", f"Metadata coverage: {meta_coverage_actual:.2%} (< 95%)"))
except Exception as e:
    gate_results.append(("G09-13", "WARN", str(e)))

# -- Parquet round-trip integrity ----------------------------------------------
try:
    assert reload_train.shape == train_features.shape, (
        f"Shape mismatch: {reload_train.shape} != {train_features.shape}"
    )
    assert reload_test.shape == test_features.shape, (
        f"Shape mismatch: {reload_test.shape} != {test_features.shape}"
    )
    assert list(reload_train.columns) == list(train_features.columns), "Column order mismatch"
    gate_results.append(("G09-RT", "PASS", "Parquet round-trip integrity verified"))
except AssertionError as e:
    gate_results.append(("G09-RT", "FAIL", str(e)))

# -- Print gate summary -------------------------------------------------------
print("=" * 70)
print("  VALIDATION GATE -- G09 RESULTS")
print("=" * 70)
for gid, status, msg in gate_results:
    if status == "PASS":
        icon = "[PASS]"
    elif "WARN" in status:
        icon = "[WARN]"
    else:
        icon = "[FAIL]"
    print(f"  {icon} {gid}: {msg}")

passed = sum(1 for _, s, _ in gate_results if s == "PASS")
warned = sum(1 for _, s, _ in gate_results if "WARN" in s)
failed = sum(1 for _, s, _ in gate_results if "FAIL" in s)
total = len(gate_results)
print(f"\n  Result: {passed}/{total} passed, {warned} warnings, {failed} failures.")
print("=" * 70)

# Halt on critical failures
critical_ids = {"G09-1", "G09-2", "G09-3", "G09-4", "G09-5", "G09-6", "G09-9", "G09-10", "G09-11", "G09-RT"}
critical_failures = [g for g in gate_results if g[1] == "FAIL" and g[0] in critical_ids]
assert len(critical_failures) == 0, f"Critical gate failures: {critical_failures}"

  VALIDATION GATE -- G09 RESULTS
  [PASS] G09-1: Cleaned data loaded with PetID and PhotoAmt columns
  [PASS] G09-2: Pretrained model loaded: efficientnet_v2_s, dim=1280
  [PASS] G09-3: No NaN values in embedding columns (zero-vector fallback active)
  [PASS] G09-4: Train and test have identical column sets
  [PASS] G09-5: Row counts preserved: 14,993 / 3,972
  [PASS] G09-6: PetID uniqueness maintained
  [PASS] G09-7: Image extraction error rate: 0.0000%
  [WARN] G09-8: PCA explained variance: 0.8274 (< 0.90)
  [PASS] G09-9: No target leakage in feature matrices
  [PASS] G09-10: Parquet files exist and are loadable
  [PASS] G09-11: schema.json exists, valid JSON, records model name
  [PASS] G09-12: has_image_embedding correctly marks 341 PetIDs with PhotoAmt==0
  [PASS] G09-13: Metadata coverage: 100.00% of PetIDs with photos
  [PASS] G09-RT: Parquet round-trip integrity verified

  Result: 13/14 passed, 1 warnings, 0 failures.


---
## 16. Feature Summary

In [28]:
# -- Feature catalog with descriptions -----------------------------------------
catalog_rows = []

# Auxiliary features
for col in IMAGE_AUX_COLUMNS:
    if col in train_features.columns:
        catalog_rows.append({
            "Feature": col,
            "Group": "auxiliary",
            "Dtype": str(train_features[col].dtype),
            "Nulls": int(train_features[col].isna().sum()),
            "Description": all_descriptions.get(col, ""),
        })

# Embeddings (summary row)
catalog_rows.append({
    "Feature": f"img_emb_0 .. img_emb_{n_emb - 1}",
    "Group": "embedding",
    "Dtype": "float32",
    "Nulls": int(train_features[emb_col_names].isna().sum().sum()),
    "Description": f"{n_emb}-dim image embeddings (efficientnet_v2_s{', PCA-reduced' if IMAGE_CONFIG['apply_pca'] else ''})",
})

# Quality features
for col in IMAGE_QUALITY_COLUMNS:
    if col in train_features.columns:
        catalog_rows.append({
            "Feature": col,
            "Group": "quality",
            "Dtype": str(train_features[col].dtype),
            "Nulls": int(train_features[col].isna().sum()),
            "Description": all_descriptions.get(col, ""),
        })

# Metadata features
for col in IMAGE_METADATA_COLUMNS:
    if col in train_features.columns:
        catalog_rows.append({
            "Feature": col,
            "Group": "metadata",
            "Dtype": str(train_features[col].dtype),
            "Nulls": int(train_features[col].isna().sum()),
            "Description": all_descriptions.get(col, ""),
        })

catalog_df = pd.DataFrame(catalog_rows)
display(
    catalog_df.style
    .set_caption("Image Feature Catalog (v1)")
    .set_properties(**{"text-align": "left"})
    .hide(axis="index")
)

Feature,Group,Dtype,Nulls,Description
has_image_embedding,auxiliary,int32,0,"Binary flag: 1 if PetID has at least one image, 0 otherwise"
actual_photo_count,auxiliary,int32,0,Number of image files found on disk for PetID
img_emb_0 .. img_emb_99,embedding,float32,0,"100-dim image embeddings (efficientnet_v2_s, PCA-reduced)"
mean_image_brightness,quality,float64,0,Mean brightness across all images for PetID (0-255 scale)
mean_blur_score,quality,float64,0,Mean Laplacian variance (blur score) across images for PetID
image_size_std,quality,float64,0,Standard deviation of image areas (width*height) across images
meta_label_count_mean,metadata,float32,0,Mean number of Vision API labels per image for PetID
meta_top_label_score_mean,metadata,float32,0,Mean top-label confidence score per image for PetID
meta_dominant_color_count_mean,metadata,float32,0,Mean dominant color count per image for PetID
meta_avg_brightness_mean,metadata,float32,0,Mean Vision API brightness proxy per image for PetID


---
## Summary

<div style="border-left: 4px solid #27ae60; padding: 10px 15px; margin: 10px 0; background: #232a3b;">

**Image Feature Extraction -- Complete**

1. **Pretrained backbone** -- `efficientnet_v2_s` (torchvision), 1,280-dimensional penultimate-layer features, 384x384 input. Classification head removed, eval mode.
2. **Batched extraction** -- All images processed in configurable batches with GPU/CPU auto-detection. Failed image loads produce zero-vector fallbacks.
3. **Per-PetID aggregation** -- Mean pooling across all images per PetID. Zero-vector for PetIDs with no images (`has_image_embedding` = 0).
4. **PCA dimensionality reduction** -- 1,280 -> 100 components (fitted on train, applied to both splits). PCA model saved for inference reproducibility.
5. **Image quality features** -- 3 features: `mean_image_brightness`, `mean_blur_score`, `image_size_std`.
6. **Vision API metadata features** -- 6 features aggregated from Google Vision API JSONs: label counts, top-label confidence, color diversity, brightness, crop confidence.

**Artifacts produced:**
- `data/features/images/v1/train.parquet` (14,993 rows)
- `data/features/images/v1/test.parquet` (3,972 rows)
- `data/features/images/v1/schema.json`
- `artifacts/models/image_pca_v1.joblib`
- `reports/feature_extraction_images.json`

</div>

**Next:** Notebook `10_feature_integration` (cross-modality feature fusion).